# Accessing the Monster catalog from the Rubin Butler main
This notebook demonstrates how to:
1. Import the project I/O helpers from ../../../src/spesn/io.py.
2. Open the Rubin Butler repository main.
3. Check that the Monster dataset is available.
4. Retrieve the HTM shards covering selected sky positions, using fetch_monster_catalog_for_sources from io.py.
5. Optionally dump all Monster shards to disk.
Important: the io.py helper was originally configured for DP1. Here we explicitly open main and pass the Monster dataset name. The Butler repository (main) and Butler collection are distinct concepts; leave COLLECTIONS = None initially to use the repository's default search path. If your installation requires a specific collection, set it in the configuration cell below.

In [ ]:
from pathlib import Path
import sys
import importlib.util

import numpy as np
import pandas as pd
from astropy.table import vstack

import lsst.daf.butler as dafButler



## 1. Import the project's io.py
The module is named io.py, which conflicts with Python's standard-library module named io. To avoid importing the wrong module, load it explicitly from its file path.

- `ìo.py` obtained from https://github.com/LSSTDESC/spesn/blob/main/spesn/io.py

In [ ]:
# Add the project helper directory to the Python import path.
IO_DIR = Path("../../src/spesn").resolve()
IO_FILE = IO_DIR / "io.py"

if not IO_FILE.is_file():
    raise FileNotFoundError(f"Cannot find the I/O helper module: {IO_FILE}")

sys.path.insert(0, str(IO_DIR))

In [ ]:
# Load io.py under a non-conflicting module name.
spec = importlib.util.spec_from_file_location("spesn_io", IO_FILE)
spesn_io = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = spesn_io
spec.loader.exec_module(spesn_io)

print(f"Loaded helper module from: {IO_FILE}")
print(f"Monster dataset configured in io.py: {spesn_io.MONSTER_DATASET}")

### 1.1 Open the Butler repository main
Set BUTLER_REPO to the Butler repository alias or path used in your environment. In the Rubin environment where main is configured as a Butler repository, the value can simply be "main".

COLLECTIONS is optional. None lets Butler use its default collection search path. If the dataset is not found, inspect the available collections and set the appropriate collection explicitly.

In [ ]:
BUTLER_REPO = "main"
COLLECTIONS = "u/dagoret/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage1"

butler = dafButler.Butler(BUTLER_REPO, collections=COLLECTIONS)

registry = butler.registry
print(f"Butler repository: {BUTLER_REPO}")
print(f"Butler instance: {butler}")

## 2. Dump the Monster dataset 

In [ ]:
# Inspect registered Monster dataset types
monster_types = sorted(
    dt.name
    for dt in butler.registry.queryDatasetTypes("*monster*")
)

print(monster_types)

In [ ]:
spesn_io.MONSTER_DATASET

In [ ]:
# Query one reference catalog dataset
refs = butler.registry.queryDatasets(
    "the_monster_20250219",
    collections="refcats",
)

ref = next(iter(refs))

print("Dataset type:", ref.datasetType)
print("Data ID:", ref.dataId)

# Retrieve catalog
monster_cat = butler.get(ref)

# Inspect schema
print(monster_cat.schema)

# Inspect column names
monster = monster_cat.asAstropy()

for name in monster.colnames:
    print(name)

## 3. Check the Monster dataset and inspect its data IDs
The helper module defines the dataset type as the_monster_20250219. Querying the registry first checks that this dataset type is present in the selected collection search path and shows the available HTM shards.

In [ ]:
MONSTER_DATASET = spesn_io.MONSTER_DATASET

# List the dataset types known to this Butler repository.
dataset_types = sorted(
    name for name in registry.queryDatasetTypes()
    if name == MONSTER_DATASET
)
print("Matching dataset types:", dataset_types)

# Retrieve dataset references for the Monster catalog.
monster_refs = list(
    registry.queryDatasets(
        MONSTER_DATASET,
        collections=COLLECTIONS,
    )
)

print(f"Number of Monster shard datasets: {len(monster_refs)}")

if not monster_refs:
    raise RuntimeError(
        f"No datasets found for {MONSTER_DATASET!r}. "
        "Check the repository and collection configuration."
    )

# Display a few data IDs to inspect the HTM sharding scheme.
for ref in monster_refs[:10]:
    print(ref.dataId)

## 4. Retrieve Monster stars for selected sky positions using io.py
The function fetch_monster_catalog_for_sources expects a pandas DataFrame with ra and dec columns in degrees. It computes the HTM level-7 IDs intersecting those positions, reads the corresponding `the_monster_20250219` shards from the Butler, and returns the concatenated rows.

Replace the example coordinates below with the sky positions of the stars or sources you want to inspect. Multiple positions can be supplied; duplicate HTM shards are fetched only once.

In [ ]:
# Example input positions (degrees): replace these with your target positions.
sources = pd.DataFrame(
    {
        "ra": [150.1191],
        "dec": [2.2058],
    }
)

monster = spesn_io.fetch_monster_catalog_for_sources(
    butler=butler,
    sources=sources,
    monster_dataset=MONSTER_DATASET,
    htm_level=7,
)

print(f"Number of Monster rows retrieved: {len(monster):,}")
print(f"Number of columns: {len(monster.columns)}")
display(monster.head())

## 5. Inspect coordinates and catalog columns
The helper converts `coord_ra` and `coord_dec` from radians to degrees and adds `coord_ra_deg` and `coord_dec_deg` when those columns are present.

In [ ]:
print("Catalog columns:")
print(monster.columns.tolist())

coordinate_columns = [
    name for name in ("coord_ra", "coord_dec", "coord_ra_deg", "coord_dec_deg")
    if name in monster.columns
]
display(monster[coordinate_columns].head())

if {"coord_ra_deg", "coord_dec_deg"}.issubset(monster.columns):
    print("RA range (deg):", monster["coord_ra_deg"].min(), monster["coord_ra_deg"].max())
    print("Dec range (deg):", monster["coord_dec_deg"].min(), monster["coord_dec_deg"].max())

## 6. Save the retrieved rows
This saves the rows fetched for the selected sky positions. Parquet is convenient for subsequent pandas analysis; CSV is also provided as an optional interchange format.

In [ ]:
# Set this to True only when you intend to dump the complete catalog.
DUMP_ALL_SHARDS = False

if DUMP_ALL_SHARDS:
    full_dump_dir = OUTPUT_DIR / "all_shards"
    full_dump_dir.mkdir(parents=True, exist_ok=True)

    for i, ref in enumerate(monster_refs, start=1):
        # Read one shard from the Butler and convert it to a pandas DataFrame.
        shard_table = butler.get(ref).asAstropy()
        shard_df = shard_table.to_pandas()

        # Add degree-valued coordinates when the catalog stores radians.
        if "coord_ra" in shard_df.columns:
            shard_df["coord_ra_deg"] = np.rad2deg(shard_df["coord_ra"])
        if "coord_dec" in shard_df.columns:
            shard_df["coord_dec_deg"] = np.rad2deg(shard_df["coord_dec"])

        # Use the HTM data ID to create a stable shard filename.
        htm7 = ref.dataId["htm7"]
        shard_path = full_dump_dir / f"the_monster_htm7_{htm7}.parquet"
        shard_df.to_parquet(shard_path, index=False)

        if i == 1 or i % 100 == 0 or i == len(monster_refs):
            print(f"[{i}/{len(monster_refs)}] Wrote {shard_path.name} "
                  f"({len(shard_df):,} rows)")

    print(f"Full shard dump written to: {full_dump_dir.resolve()}")
else:
    print("Full catalog dump skipped. Set DUMP_ALL_SHARDS = True to run it.")

## Notes
- fetch_monster_catalog_for_sources is the appropriate entry point when you need the Monster stars around a set of known sky positions.
- The full-dump cell reads every HTM shard directly through Butler references. It does not concatenate the full catalog in memory.
- If queryDatasets returns no references, first verify that main is the correct repository alias and then set COLLECTIONS to the collection containing the_monster_20250219.
- If the dataset uses a different HTM dimension name, inspect ref.dataId in Section 3 and adapt the htm7 access in the full-dump cell.